In [ ]:
import logging

logging.basicConfig(
    filename="day2.log",
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("internship_day2")
print("Logging configured -> day2.log")


Logging configured -> day2.log


## Load Data

In [2]:
import pandas as pd

file_path = "data/Test.csv"

df = pd.read_csv(file_path)

logger.info("Loaded %s | shape=%s", file_path, df.shape)
print(df.shape)
print(df.head())

(30, 10)
   Customer_ID  Customer_Name                    Email       City  Product  \
0         2001   Suman Basnet   suman.basnet@email.com  Kathmandu   Laptop   
1         2002   Anita Poudel   anita.poudel@email.com        ktm    Phone   
2         2003  Bishal Tamang  bishal.tamang@email.com  KATHMANDU   Laptop   
3         2004       Gita Rai           gita.rai@email   Lalitpur  Tablet    
4         2005       Ram K.C.                      NaN  Bhaktapur    phone   

   Quantity  Unit_Price  Order_Date  Gender  Age  
0         2       750.0    5-Jan-26    Male   25  
1         1       450.0    7-Jan-26  Female   32  
2         1       800.0  2026-01-09    Male   28  
3         3       300.0   10-Jan-26  Female   41  
4         2       500.0   11-Jan-26    Male   35  


## Identifying Missing Values

In [3]:
print("Missing values per column:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nData types:")
print(df.dtypes)

logger.info(
    "Data quality check | missing_total=%s | duplicates=%s",
    df.isnull().sum().sum(),
    df.duplicated().sum(),
)

Missing values per column:
Customer_ID      0
Customer_Name    0
Email            2
City             1
Product          0
Quantity         0
Unit_Price       1
Order_Date       0
Gender           0
Age              0
dtype: int64

Duplicate rows: 1

Data types:
Customer_ID        int64
Customer_Name        str
Email                str
City                 str
Product              str
Quantity           int64
Unit_Price       float64
Order_Date           str
Gender               str
Age                int64
dtype: object


## Drop Duplicate data

In [4]:
before = len(df)
df = df.drop_duplicates()
after = len(df)

logger.info("Removed %s duplicate row(s) | rows: %s -> %s", before - after, before, after)
print(f"Dropped {before - after} duplicate row(s). Rows remaining: {after}")

Dropped 1 duplicate row(s). Rows remaining: 29


In [5]:
for col in ["City", "Product", "Gender"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["City"] = df["City"].replace({"Ktm": "Kathmandu"})
df["Gender"] = df["Gender"].replace({"M": "Male", "F": "Female"})

logger.info("Standardized categorical columns: City, Product, Gender")
print(df[["City", "Product", "Gender"]].nunique())

City       4
Product    4
Gender     2
dtype: int64


## Filling Missing Value

In [6]:
# Numeric columns: fill with median
for col in ["Unit_Price", "Age"]:
    median_val = df[col].median()
    missing = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    if missing:
        logger.info("Filled %s missing value(s) in %s with median=%s", missing, col, median_val)

# Categorical columns: fill with an explicit placeholder rather than guessing
df["City"] = df["City"].fillna("Unknown")
df["Email"] = df["Email"].fillna("unknown@example.com")

logger.info("Handled missing values in Unit_Price, Age, City, Email")
print(df.isnull().sum())

Customer_ID      0
Customer_Name    0
Email            0
City             0
Product          0
Quantity         0
Unit_Price       0
Order_Date       0
Gender           0
Age              0
dtype: int64


In [7]:
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce", format="mixed")
invalid_dates = df["Order_Date"].isnull().sum()

logger.info("Converted Order_Date to datetime | invalid_dates=%s", invalid_dates)
print(f"Invalid/unparseable dates: {invalid_dates}")
print(df["Order_Date"].head())

Invalid/unparseable dates: 1
0   2026-01-05
1   2026-01-07
2   2026-01-09
3   2026-01-10
4   2026-01-11
Name: Order_Date, dtype: datetime64[us]


In [8]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ["Age", "Unit_Price"]:
    lower, upper = iqr_bounds(df[col])
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    logger.info(
        "Capped %s outlier(s) in %s to range [%.2f, %.2f]", outliers, col, lower, upper
    )
    print(f"{col}: capped {outliers} outlier(s) to [{lower:.2f}, {upper:.2f}]")

Age: capped 2 outlier(s) to [9.00, 57.00]
Unit_Price: capped 1 outlier(s) to [-417.50, 1522.50]


In [9]:
df["Total_Amount"] = df["Quantity"] * df["Unit_Price"]

logger.info("Added Total_Amount = Quantity * Unit_Price")
print(df[["Quantity", "Unit_Price", "Total_Amount"]].head())

   Quantity  Unit_Price  Total_Amount
0         2       750.0        1500.0
1         1       450.0         450.0
2         1       800.0         800.0
3         3       300.0         900.0
4         2       500.0        1000.0


In [10]:
output_path = "data/Test_cleaned.csv"
df.to_csv(output_path, index=False)

logger.info("Saved cleaned data -> %s | shape=%s", output_path, df.shape)
print(f"Saved cleaned data to {output_path}")
print(df.shape)
df.head()

Saved cleaned data to data/Test_cleaned.csv
(29, 11)


,Customer_ID,Customer_Name,Email,City,Product,Quantity,Unit_Price,Order_Date,Gender,Age,Total_Amount
0,2001,Suman Basnet,suman.basnet@email.com,Kathmandu,Laptop,2,750.0,2026-01-05,Male,25,1500.0
1,2002,Anita Poudel,anita.poudel@email.com,Kathmandu,Phone,1,450.0,2026-01-07,Female,32,450.0
2,2003,Bishal Tamang,bishal.tamang@email.com,Kathmandu,Laptop,1,800.0,2026-01-09,Male,28,800.0
3,2004,Gita Rai,gita.rai@email,Lalitpur,Tablet,3,300.0,2026-01-10,Female,41,900.0
4,2005,Ram K.C.,unknown@example.com,Bhaktapur,Phone,2,500.0,2026-01-11,Male,35,1000.0


In [11]:
logger.info("Day 2 completed successfully")